# 第六章 大模型训练流程实践 · 代码汇总

> 原文：[第六章 大模型训练流程实践.md](./第六章%20大模型训练流程实践.md)  
> 参考代码：[`code/`](./code)

本 Notebook 按原文顺序汇总第 6 章的 Python / JSON / Shell 源码，保留原有中文注释，并对 **模型初始化、数据打包、SFT loss mask、DeepSpeed 配置、LoRA 低秩旁路** 等关键部分补充了说明。

**运行说明**

- 带 `⚠️` 标记的代码需要下载 Qwen2.5-1.5B、准备数据集，或依赖 `deepspeed`、`torchdata`、`swanlab`、`peft`，默认不建议直接运行。
- 完整训练脚本保留原始 `main()` 结构，并通过 `RUN_PRETRAIN` / `RUN_TRAINING` 开关防止误启动。
- 模型前向、LoRA 自定义层等轻量代码可以直接运行。

## 0. 环境依赖

`code/requirements.txt`：

```text
transformers
datasets
torch
torchdata==0.9.0
deepspeed
pandas
swanlab
```

安装命令：

```bash
pip install -r requirements.txt
```

其中 `torchdata==0.9.0` 用于 `IterableWrapper`，`deepspeed` 用于 ZeRO 分布式训练，`swanlab` 用于训练日志。

In [ ]:
# 轻量示例只依赖 torch / transformers；训练脚本的依赖见上方 requirements.txt
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)

## 6.1 模型预训练

### 6.1.1 三个框架的分工

| 框架 | 作用 |
| --- | --- |
| Transformers | 统一模型结构、`AutoModel` 加载、`Trainer` 训练循环、数据集与评估工具 |
| DeepSpeed | ZeRO 分片、混合精度、梯度累积、优化器状态卸载，降低大模型显存占用 |
| SwanLab | 记录 loss、学习率、吞吐等训练指标，便于对比实验 |

本章用 Qwen2.5-1.5B 的架构做示例：既可以用它的 `config.json` **从零初始化**，也可以加载官方权重做**继续预训练 / SFT**。

### 6.1.2 初始化 LLM

关键概念：

- `AutoConfig.from_pretrained(...)` 只加载 **模型结构配置**（层数、隐藏维度、头数等），不加载权重。
- `AutoModelForCausalLM.from_config(config)` 按配置随机初始化一个模型，适合从头预训练。
- `AutoModelForCausalLM.from_pretrained(path)` 加载已有权重，适合继续预训练或 SFT。
- `torch_dtype="auto"` 会读取权重文件中的精度（通常是 bfloat16）。

> 原文示例中的 `model_name_or_path` 是占位变量，实际使用时请替换为上一步下载的本地目录（如 `qwen-1.5b`）。

⚠️ 以下单元格会访问 HuggingFace / ModelScope 下载配置或权重，请按需执行。

In [ ]:
import os
# 设置环境变量，此处使用 HuggingFace 镜像网站
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 下载模型
os.system('huggingface-cli download --resume-download Qwen/Qwen2.5-1.5B --local-dir your_local_dir')

In [ ]:
# 加载定义好的模型参数-此处以 Qwen-2.5-1.5B 为例
# 使用 transforemrs 的 Config 类进行加载
from transformers import AutoConfig

# 下载参数的本地路径
model_path = "qwen-1.5b"
config = AutoConfig.from_pretrained(model_name_or_path)

In [ ]:
# 使用该配置生成一个定义好的模型
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_config(config,trust_remote_code=True)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name_or_path,trust_remote_code=True)

In [ ]:
# 加载一个预训练好的 tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

### 6.1.3 预训练数据处理

预训练数据的处理分为三步：

1. **加载 JSONL**：每行一个 `{{"text": ...}}` 样本；
2. **Tokenize**：把文本转成 `input_ids`；
3. **拼接 + 切块（packing）**：把所有 token 首尾相接，再按 `block_size` 切成等长片段，减少 padding、提高训练效率。

`labels` 直接复制 `input_ids`，由因果语言模型内部错位一位计算下一个 token 的损失。

In [ ]:
# 加载预训练数据
from datasets import load_dataset

ds = load_dataset('json', data_files='/mobvoi_seq_monkey_general_open_corpus.jsonl')

In [ ]:
# 查看特征
column_names = list(ds["train"].features)
# columnes_name:["text"]

In [ ]:
# 对数据集进行 tokenize
def tokenize_function(examples):
    # 使用预先加载的 tokenizer 进行分词
    output = tokenizer([item for item in examples["text"]])
    return output

# 批量处理
tokenized_datasets = ds.map(
    tokenize_function,
    batched=True,
    num_proc=10,
    remove_columns=column_names,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)

In [ ]:
# 预训练一般将文本拼接成固定长度的文本段
from itertools import chain

# 这里我们取块长为 2048
block_size = 2048

def group_texts(examples):
    # 将文本段拼接起来
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    # 计算拼起来的整体长度
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # 如果长度太长，进行分块
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # 按 block_size 进行切分
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # CLM 任务，labels 和 input 是相同的
    result["labels"] = result["input_ids"].copy()
    return result

# 批量处理
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=10,
    load_from_cache_file=True,
    desc=f"Grouping texts in chunks of {block_size}",
    batch_size = 40000,
)
train_dataset = lm_datasets["train"]

`group_texts` 的核心是：

- 用 `chain(*examples[k])` 把同一 batch 内所有样本拼成一条长序列；
- 截断到 `block_size` 的整数倍；
- 再按 `block_size` 切片；
- `labels = input_ids.copy()`，训练时由模型负责 shift。

这样每条样本都是满的 `block_size`，不会因为 padding 浪费算力。

### 6.1.4 使用 Trainer 训练

`TrainingArguments` 负责 batch size、学习率、日志、保存策略、混合精度等；`Trainer` 把模型、数据、collator 和训练参数组装起来，自动完成前向、反向、梯度累积、日志与 checkpoint 保存。

关键参数：

- `per_device_train_batch_size`：单卡 batch；
- `gradient_accumulation_steps`：梯度累积，等效放大 batch；
- `learning_rate`、`warmup_steps`、`num_train_epochs`；
- `bf16=True`：使用 bfloat16 混合精度；
- `gradient_checkpointing=True`：用计算换显存；
- `deepspeed=...`：接入 DeepSpeed 配置。

In [ ]:
from transformers import TrainingArguments
# 配置训练参数

training_args = TrainingArguments(
    output_dir="output",# 训练参数输出路径
    per_device_train_batch_size=4,# 训练的 batch_size
    gradient_accumulation_steps=4,# 梯度累计步数，实际 bs = 设置的 bs * 累计步数
    logging_steps=10,# 打印 loss 的步数间隔
    num_train_epochs=1,# 训练的 epoch 数
    save_steps=100, # 保存模型参数的步数间隔
    learning_rate=1e-4,# 学习率
    gradient_checkpointing=True# 开启梯度检查点
)

In [ ]:
from transformers import Trainer, default_data_collator
from torchdata.datapipes.iter import IterableWrapper

# 训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset= IterableWrapper(train_dataset),
    eval_dataset= None,
    tokenizer=tokenizer,
    # 默认为 MLM 的 collator，使用 CLM 的 collater
    data_collator=default_data_collator
)

In [ ]:
trainer.train()

### 6.1.5 使用 DeepSpeed 实现分布式训练

DeepSpeed ZeRO 的核心思想是把**模型参数、梯度、优化器状态**切分到多张卡上：

- **Stage 1**：切分优化器状态；
- **Stage 2**：再切分梯度（本配置使用 Stage 2）；
- **Stage 3**：再切分模型参数。

`ds_config_zero2.json` 的关键项：

- `bf16.enabled = auto`：由启动参数决定是否启用 bf16；
- `zero_optimization.stage = 2`：梯度分片；
- `offload_optimizer.device = none`：不把优化器状态卸载到 CPU；
- `allgather_partitions` / `reduce_scatter` / `overlap_comm`：通信与显存优化；
- `train_batch_size` 等设为 `auto`，由 HuggingFace `TrainingArguments` 自动填充。

DeepSpeed 配置 `ds_config_zero2.json`：

```json
{
    "fp16": {
        "enabled": "auto",
        "loss_scale": 0,
        "loss_scale_window": 1000,
        "initial_scale_power": 16,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "bf16": {
        "enabled": "auto"
    },
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": "auto",
            "betas": "auto",
            "eps": "auto",
            "weight_decay": "auto"
        }
    },

    "scheduler": {
        "type": "WarmupLR",
        "params": {
            "warmup_min_lr": "auto",
            "warmup_max_lr": "auto",
            "warmup_num_steps": "auto"
        }
    },

    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "none",
            "pin_memory": true
        },
        "allgather_partitions": true,
        "allgather_bucket_size": 2e8,
        "overlap_comm": true,
        "reduce_scatter": true,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": true
    },

    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "steps_per_print": 100,
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "wall_clock_breakdown": false
}
```

启动脚本 `pretrain.sh`：

```bash
CUDA_VISIBLE_DEVICES=0,1

deepspeed pretrain.py \
    --config_name autodl-tmp/qwen-1.5b \
    --tokenizer_name autodl-tmp/qwen-1.5b \
    --train_files autodl-tmp/dataset/pretrain_data/mobvoi_seq_monkey_general_open_corpus_small.jsonl \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 4 \
    --do_train \
    --output_dir autodl-tmp/output/pretrain \
    --evaluation_strategy  no \
    --learning_rate 1e-4 \
    --num_train_epochs 1 \
    --warmup_steps 200 \
    --logging_dir autodl-tmp/output/pretrain/logs \
    --logging_strategy steps \
    --logging_steps 5 \
    --save_strategy steps \
    --save_steps 100 \
    --preprocessing_num_workers 10 \
    --save_total_limit 1 \
    --seed 12 \
    --block_size 2048 \
    --bf16 \
    --gradient_checkpointing \
    --deepspeed ./ds_config_zero2.json \
    --report_to swanlab
    
    # --resume_from_checkpoint ${output_model}/checkpoint-20400 \
```

要点：`--deepspeed ./ds_config_zero2.json` 启用 ZeRO-2；`--gradient_checkpointing` 降低显存；`--bf16` 使用 bfloat16；`--resume_from_checkpoint` 可从断点恢复。

下面是完整的 `pretrain.py`。与原文 6.1.5 节的分段代码相比，这里保留了全部导入、参数类和 `main()`，便于整体阅读。

⚠️ 依赖 `deepspeed`、`torchdata`、`swanlab`，并需要 GPU 与数据；`RUN_PRETRAIN=False` 时不会启动训练。

In [ ]:
# ⚠️ 完整训练脚本：未安装 deepspeed/torchdata/swanlab 时请勿运行本单元格
RUN_PRETRAIN = False  # 改为 True 并准备好模型/数据/GPU 后再运行

'''
预训练脚本
'''

import logging
import math
import os
import sys
from dataclasses import dataclass, field
from torchdata.datapipes.iter import IterableWrapper
from itertools import chain
import deepspeed
from typing import Optional,List

import datasets
import pandas as pd
import torch
from datasets import load_dataset
import transformers
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
import datetime
from transformers.testing_utils import CaptureLogger
from transformers.trainer_utils import get_last_checkpoint
import swanlab


logger = logging.getLogger(__name__)


# 超参类
@dataclass
class ModelArguments:
    """
    关于模型的参数
    """

    model_name_or_path: Optional[str] = field(
        default=None,
        metadata={
            "help": (
                "后训练使用，为预训练模型参数地址"
            )
        },
    )
    config_name: Optional[str] = field(
        default=None, metadata={"help": "预训练使用，Config 文件地址"}
    )
    tokenizer_name: Optional[str] = field(
        default=None, metadata={"help": "预训练 Tokenizer 地址"}
    )
    torch_dtype: Optional[str] = field(
        default=None,
        metadata={
            "help": (
                "模型训练使用的数据类型，推荐 bfloat16"
            ),
            "choices": ["auto", "bfloat16", "float16", "float32"],
        },
    )


@dataclass
class DataTrainingArguments:
    """
    关于训练的参数
    """

    train_files: Optional[List[str]]  = field(default=None, metadata={"help": "训练数据路径"})
    block_size: Optional[int] = field(
        default=None,
        metadata={
            "help": (
                "设置的文本块长度"
            )
        },
    )
    preprocessing_num_workers: Optional[int] = field(
        default=None,
        metadata={"help": "预处理使用线程数."},
    )

                
def main():

    # 加载脚本参数
    parser = HfArgumentParser((ModelArguments, DataTrainingArguments, TrainingArguments))
    model_args, data_args, training_args = parser.parse_args_into_dataclasses()

    # 初始化 SwanLab
    swanlab.init(project="pretrain", experiment_name="from_scrach")
    
    # 设置日志
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        handlers=[logging.StreamHandler(sys.stdout)],
    )

    # 将日志级别设置为 INFO
    transformers.utils.logging.set_verbosity_info()
    log_level = training_args.get_process_log_level()
    logger.setLevel(log_level)
    datasets.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.enable_default_handler()
    transformers.utils.logging.enable_explicit_format()

    # 训练整体情况记录
    logger.warning(
        f"Process rank: {training_args.local_rank}, device: {training_args.device}, n_gpu: {training_args.n_gpu}"
        + f"distributed training: {bool(training_args.local_rank != -1)}, 16-bits training: {training_args.fp16}"
    )
    logger.info(f"Training/evaluation parameters {training_args}")

    # 检查 checkpoint
    last_checkpoint = None
    if os.path.isdir(training_args.output_dir):
        last_checkpoint = get_last_checkpoint(training_args.output_dir)
        if last_checkpoint is None and len(os.listdir(training_args.output_dir)) > 0:
            raise ValueError(
                f"输出路径 ({training_args.output_dir}) 非空 "
            )
        elif last_checkpoint is not None and training_args.resume_from_checkpoint is None:
            logger.info(
                f"从 {last_checkpoint}恢复训练"
            )

    # 设置随机数种子.
    set_seed(training_args.seed)

    # 初始化模型
    if model_args.config_name is not None:
        # from scrach
        config = AutoConfig.from_pretrained(model_args.config_name)
        logger.warning("你正在从零初始化一个模型")
        logger.info(f"模型参数配置地址：{model_args.config_name}")
        logger.info(f"模型参数：{config}")
        model = AutoModelForCausalLM.from_config(config,trust_remote_code=True)
        n_params = sum({p.data_ptr(): p.numel() for p in model.parameters()}.values())
        logger.info(f"预训练一个新模型 - Total size={n_params/2**20:.2f}M params")
    elif model_args.model_name_or_path is not None:
        logger.warning("你正在初始化一个预训练模型")
        logger.info(f"模型参数地址：{model_args.model_name_or_path}")
        model = AutoModelForCausalLM.from_pretrained(model_args.model_name_or_path,trust_remote_code=True)
        n_params = sum({p.data_ptr(): p.numel() for p in model.parameters()}.values())
        logger.info(f"继承一个预训练模型 - Total size={n_params/2**20:.2f}M params")
    else:
        logger.error("config_name 和 model_name_or_path 不能均为空")
        raise ValueError("config_name 和 model_name_or_path 不能均为空")

    # 初始化 Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_args.tokenizer_name)
    logger.info("完成 tokenzier 加载")
    logger.info(f"tokenzier 配置地址：{model_args.tokenizer_name}")

    # 加载预训练数据
    ds = load_dataset('json', data_files=data_args.train_files)
    logger.info("完成训练集加载")
    logger.info(f"训练集地址：{data_args.train_files}")
    logger.info(f'训练文件总数:{len(ds["train"])}')
    # logger.info(f"训练集采样：{ds["train"][0]}")

    # 文本 tokenize
    column_names = list(ds["train"].features)
    logger.info('训练集特征：', column_names)
    text_column_name = "text" if "text" in column_names else column_names[0]

    # tokenize 函数
    def tokenize_function(examples):
        output = tokenizer([item for item in examples[text_column_name]])
        return output

    # 仅主进程进行数据预处理
    with training_args.main_process_first(desc="dataset map tokenization"):
        tokenized_datasets = ds.map(
            tokenize_function,
            batched=True,
            num_proc=data_args.preprocessing_num_workers,
            remove_columns=column_names,
            load_from_cache_file=True,
            desc="Running tokenizer on dataset"
        )

    # 文本切块
    if data_args.block_size is None:
        block_size = tokenizer.model_max_length
        if block_size > 1024:
            logger.warning(
                "tokenizer 支持大于 1K 的上下文长度，默认设置为 1K"
            )
            block_size = 1024
    else:
        if data_args.block_size > tokenizer.model_max_length:
            logger.warning(
                f"设定的块长为 ({data_args.block_size}) ，大于模型的上下文长度"
                f"将块长设置为模型上下文长度：{tokenizer.model_max_length}."
            )
        block_size = min(data_args.block_size, tokenizer.model_max_length)

    def group_texts(examples):
        # 将文本段拼接起来
        concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
        # 计算拼起来的整体长度
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # 如果长度太长，进行分块
        if total_length >= block_size:
            total_length = (total_length // block_size) * block_size
        result = {
            k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
            for k, t in concatenated_examples.items()
        }    
        result["labels"] = result["input_ids"].copy()
        return result

    with training_args.main_process_first(desc="文本分块"):
        lm_datasets = tokenized_datasets.map(
            group_texts,
            batched=True,
            num_proc=data_args.preprocessing_num_workers,
            load_from_cache_file=True,
            desc=f"文本分块到{block_size}",
            batch_size = 40000,
        )
        logger.info("完成数据预处理")
        train_dataset = lm_datasets["train"]
    
    logger.info("初始化 Trainer")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset= IterableWrapper(train_dataset),
        tokenizer=tokenizer,
        data_collator=default_data_collator
    )

    # 从 checkpoint 加载
    checkpoint = None
    if training_args.resume_from_checkpoint is not None:
        checkpoint = training_args.resume_from_checkpoint
    elif last_checkpoint is not None:
            checkpoint = last_checkpoint

    logger.info("开始训练")
    train_result = trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model() 

if __name__ == "__main__" and RUN_PRETRAIN:
    main()

## 6.2 模型有监督微调（SFT）

### 6.2.1 Pretrain 与 SFT 的差异

| 阶段 | 数据 | 损失范围 | 目标 |
| --- | --- | --- | --- |
| Pretrain | 海量无监督文本 | 全部 token | 学习语言规律与世界知识 |
| SFT | 指令-回答对 | 只计算 assistant 回复 | 学会遵循指令、按格式回答 |

两者都使用因果语言模型（CLM），因此训练循环几乎一致，**核心差异只在数据构造和 loss mask**。

### 6.2.2 微调数据处理

Qwen 的 Chat Template 把多轮对话拼成如下序列：

```text
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>human
用户问题<|im_end|>
<|im_start|>assistant
模型回答<|im_end|>
```

关键点：

- `system` 和 `human` 部分**不计算损失**，对应 target 用 `IGNORE_TOKEN_ID` 填充；
- `assistant` 部分才参与损失计算；
- 多轮对话按顺序拼接，最后统一 padding 到 `max_len`；
- `attention_mask` 标记真实 token 与 padding。

原文定义特殊 token 的片段：

```python
# 不同的 tokenizer 需要特别定义
im_start = tokenizer("<|im_start|>").input_ids  # BOS
im_end = tokenizer("<|im_end|>").input_ids      # EOS
IGNORE_TOKEN_ID = tokenizer.pad_token_id        # PAD
nl_tokens = tokenizer('\n').input_ids          # 换行符
_system = tokenizer('system').input_ids + nl_tokens
_user = tokenizer('human').input_ids + nl_tokens
_assistant = tokenizer('assistant').input_ids + nl_tokens
```

In [ ]:
import json
from typing import Dict
from tqdm import tqdm

In [ ]:
# 指令文本处理
# 参考：https://github.com/QwenLM/Qwen/blob/main/finetune.py
def preprocess(sources, tokenizer, max_len, system_message: str = "You are a helpful assistant."):
    # prompt 模板
    roles = {"human": "<|im_start|>human", "assistant": "<|im_start|>assistant"}

    # 不同的 tokenizer 需要特别定义
    # BOS
    im_start = tokenizer("<|im_start|>").input_ids
    # EOS
    im_end = tokenizer("<|im_end|>").input_ids
    # PAD
    IGNORE_TOKEN_ID = tokenizer.pad_token_id
    # 换行符
    nl_tokens = tokenizer('\n').input_ids
    # 角色标识符
    _system = tokenizer('system').input_ids + nl_tokens
    _user = tokenizer('human').input_ids + nl_tokens
    _assistant = tokenizer('assistant').input_ids + nl_tokens

    # 拼接多轮对话
    input_ids, targets = [], []
    for i in tqdm(range(len(sources))):
        source = sources[i]
        # 从 user 开始
        if source[0]["from"] != "human":
            source = source[1:]
        # 分别是输入和输出
        input_id, target = [], []
        # system: 【BOS】system\nYou are a helpful assistant.【EOS】\n
        system = im_start + _system + tokenizer(system_message).input_ids + im_end + nl_tokens
        input_id += system
        # system 不需要拟合
        target += im_start + [IGNORE_TOKEN_ID] * (len(system)-3) + im_end + nl_tokens
        assert len(input_id) == len(target)
        # 依次拼接
        for j, sentence in enumerate(source):
            role = roles[sentence["from"]]
            # user：<|im_start|>human\ninstruction【EOS】\n
            # assistant：<|im_start|>assistant\nresponse【EOS】\n
            _input_id = tokenizer(role).input_ids + nl_tokens + \
                tokenizer(sentence["value"]).input_ids + im_end + nl_tokens
            input_id += _input_id
            if role == '<|im_start|>human':
                # user 不需要拟合
                _target = im_start + [IGNORE_TOKEN_ID] * (len(_input_id)-3) + im_end + nl_tokens
            elif role == '<|im_start|>assistant':
                # assistant 需要拟合
                _target = im_start + [IGNORE_TOKEN_ID] * len(tokenizer(role).input_ids) + \
                    _input_id[len(tokenizer(role).input_ids)+1:-2] + im_end + nl_tokens
            else:
                print(role)
                raise NotImplementedError
            target += _target
        assert len(input_id) == len(target)
        # 最后进行 PAD
        input_id += [tokenizer.pad_token_id] * (max_len - len(input_id))
        target += [IGNORE_TOKEN_ID] * (max_len - len(target))
        input_ids.append(input_id[:max_len])
        targets.append(target[:max_len])
    # print(input_ids)
    input_ids = torch.tensor(input_ids)
    targets = torch.tensor(targets)

    return dict(
        input_ids=input_ids,
        labels=targets,
        attention_mask=input_ids.ne(tokenizer.pad_token_id),
    )

`preprocess` 的 target 构造逻辑：

- `system`：`im_start + [IGNORE] * (len(system)-3) + im_end + nl`，只保留首尾特殊 token，内容不拟合；
- `human`：同样用 `IGNORE_TOKEN_ID` 遮蔽；
- `assistant`：保留回复正文，只有角色标记部分被遮蔽；
- 最后按 `max_len` 截断/补齐，并生成 `attention_mask`。

> 原文提到主流实现常把 `IGNORE_TOKEN_ID` 设为 `-100`；本脚本使用 `tokenizer.pad_token_id`，只要与 `CrossEntropyLoss(ignore_index=...)` 保持一致即可。

In [ ]:
class SupervisedDataset(Dataset):

    def __init__(self, raw_data, tokenizer, max_len: int):
        super(SupervisedDataset, self).__init__()
        # 加载并预处理数据
        sources = [example["conversations"] for example in raw_data]
        data_dict = preprocess(sources, tokenizer, max_len)

        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        self.attention_mask = data_dict["attention_mask"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(
            input_ids=self.input_ids[i],
            labels=self.labels[i],
            attention_mask=self.attention_mask[i],
        )

### 6.2.3 SFT 训练

SFT 的训练循环与预训练一致，主要差别是：

- 从预训练权重初始化，而不是 `from_config`；
- 数据使用 `SupervisedDataset`，返回 `input_ids / labels / attention_mask`；
- 只读取前 10000 条样本做示例（`f.readlines()[:10000]`）；
- 用 `Trainer` + DeepSpeed 启动。

In [ ]:
# ⚠️ 完整 SFT 训练脚本：依赖 deepspeed/torchdata/swanlab，并需要 GPU 与数据

import logging
import os
import sys
from dataclasses import dataclass, field
from typing import Optional, List

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from torchdata.datapipes.iter import IterableWrapper

try:
    import swanlab
except ImportError:
    swanlab = None

import datasets
import transformers

RUN_TRAINING = False  # 改为 True 并准备好模型/数据/GPU 后再运行

# 超参类
@dataclass
class ModelArguments:
    """
    关于模型的参数
    """

    model_name_or_path: Optional[str] = field(
        default=None,
        metadata={
            "help": (
                "预训练模型参数地址"
            )
        },
    )
    torch_dtype: Optional[str] = field(
        default=None,
        metadata={
            "help": (
                "模型训练使用的数据类型，推荐 bfloat16"
            ),
            "choices": ["auto", "bfloat16", "float16", "float32"],
        },
    )


@dataclass
class DataTrainingArguments:
    """
    关于训练的参数
    """

    train_files: Optional[str]  = field(default=None, metadata={"help": "训练数据路径"})
    block_size: Optional[int] = field(
        default=None,
        metadata={
            "help": (
                "最大文本块长度"
            )
        },
    )

def main():

    # 加载脚本参数
    parser = HfArgumentParser((ModelArguments, DataTrainingArguments, TrainingArguments))
    model_args, data_args, training_args = parser.parse_args_into_dataclasses()

    # 初始化 SwanLab
    swanlab.init(project="sft", experiment_name="qwen-1.5b")
    
    # 设置日志
    logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        handlers=[logging.StreamHandler(sys.stdout)],
    )

    # 将日志级别设置为 INFO
    transformers.utils.logging.set_verbosity_info()
    log_level = training_args.get_process_log_level()
    logger.setLevel(log_level)
    datasets.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.set_verbosity(log_level)
    transformers.utils.logging.enable_default_handler()
    transformers.utils.logging.enable_explicit_format()

    # 训练整体情况记录
    logger.warning(
        f"Process rank: {training_args.local_rank}, device: {training_args.device}, n_gpu: {training_args.n_gpu}"
        + f"distributed training: {bool(training_args.local_rank != -1)}, 16-bits training: {training_args.fp16}"
    )
    logger.info(f"Training/evaluation parameters {training_args}")

    # 检查 checkpoint
    last_checkpoint = None
    if os.path.isdir(training_args.output_dir):
        last_checkpoint = get_last_checkpoint(training_args.output_dir)
        if last_checkpoint is None and len(os.listdir(training_args.output_dir)) > 0:
            raise ValueError(
                f"输出路径 ({training_args.output_dir}) 非空 "
            )
        elif last_checkpoint is not None and training_args.resume_from_checkpoint is None:
            logger.info(
                f"从 {last_checkpoint}恢复训练"
            )

    # 设置随机数种子.
    set_seed(training_args.seed)

    # 初始化模型
    logger.warning("加载预训练模型")
    logger.info(f"模型参数地址：{model_args.model_name_or_path}")
    model = AutoModelForCausalLM.from_pretrained(model_args.model_name_or_path,trust_remote_code=True)
    n_params = sum({p.data_ptr(): p.numel() for p in model.parameters()}.values())
    logger.info(f"继承一个预训练模型 - Total size={n_params/2**20:.2f}M params")

    # 初始化 Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path)
    logger.info("完成 tokenzier 加载")

    # 加载微调数据
    with open(data_args.train_files) as f:
        lst = [json.loads(line) for line in f.readlines()[:10000]]
    logger.info("完成训练集加载")
    logger.info(f"训练集地址：{data_args.train_files}")
    logger.info(f'训练样本总数:{len(lst)}')
    # logger.info(f"训练集采样：{ds["train"][0]}")

    train_dataset = SupervisedDataset(lst, tokenizer=tokenizer, max_len=2048)
    
    logger.info("初始化 Trainer")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset= IterableWrapper(train_dataset),
        tokenizer=tokenizer
    )

    # 从 checkpoint 加载
    checkpoint = None
    if training_args.resume_from_checkpoint is not None:
        checkpoint = training_args.resume_from_checkpoint
    elif last_checkpoint is not None:
            checkpoint = last_checkpoint

    logger.info("开始训练")
    train_result = trainer.train(resume_from_checkpoint=checkpoint)
    trainer.save_model() 

if __name__ == "__main__" and RUN_TRAINING:
    main()

启动脚本 `finetune.sh`：

```bash
CUDA_VISIBLE_DEVICES=0,1

deepspeed finetune.py \
    --model_name_or_path autodl-tmp/qwen-1.5b \
    --train_files autodl-tmp/dataset/sft_data/BelleGroup/train_3.5M_CN.json \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 4 \
    --do_train \
    --output_dir autodl-tmp/output/sft \
    --evaluation_strategy  no \
    --learning_rate 1e-4 \
    --num_train_epochs 3 \
    --warmup_steps 200 \
    --logging_dir autodl-tmp/output/sft/logs \
    --logging_strategy steps \
    --logging_steps 5 \
    --save_strategy steps \
    --save_steps 100 \
    --save_total_limit 1 \
    --seed 12 \
    --block_size 2048 \
    --bf16 \
    --gradient_checkpointing \
    --deepspeed ./ds_config_zero2.json \
    --report_to swanlab
    
    # --resume_from_checkpoint ${output_model}/checkpoint-20400 \
```

## 6.3 高效微调

### 6.3.1 全量微调之外的两条路线

| 方法 | 思路 | 主要问题 |
| --- | --- | --- |
| Adapter Tuning | 每层插入小型前馈模块，只训练 Adapter | 增加层数与推理延迟 |
| Prefix / P-Tuning | 在输入前加可训练 virtual tokens | 占用序列长度 |
| LoRA | 冻结原权重，旁路低秩矩阵 `BA` | 难以注入全新知识 |

### 6.3.2 LoRA 的核心

对预训练权重 `W0 ∈ R^{d×k}`，LoRA 不直接更新 `W0`，而是学习一个低秩增量：

$$
W = W_0 + \Delta W = W_0 + BA,\quad B\in R^{d×r},\ A\in R^{r×k},\ r \ll \min(d,k)
$$

前向：

$$
h = W_0x + BAx
$$

- `A` 随机高斯初始化，`B` 零初始化，因此训练开始时 `BA = 0`，不破坏原模型；
- 缩放系数 `scaling = lora_alpha / r`；
- 训练只更新 `A`、`B`，原权重 `requires_grad=False`；
- 推理时可以把 `BA` 加回 `W0`，**不增加推理延迟**；
- 常作用于注意力的 `q_proj`、`v_proj`。

### 6.3.3 简化实现

下面把原文拆分展示的 `LoraLayer`、`Linear` 与 `forward` 合并成一个可运行的最小实现，并补上原文省略的 `transpose` 与 `reset_parameters`。

In [ ]:
def transpose(weight, fan_in_fan_out):
    """当权重按 (in, out) 存储时，需要转置后再做 F.linear。"""
    return weight.T if fan_in_fan_out else weight


class LoraLayer:
    """保存 LoRA 超参，不直接继承 nn.Module。"""

    def __init__(self, r: int, lora_alpha: int, lora_dropout: float, merge_weights: bool):
        self.r = r
        self.lora_alpha = lora_alpha
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else (lambda x: x)
        self.merged = False          # 是否已把 BA 合并进原权重
        self.merge_weights = merge_weights
        self.disable_adapters = False


class Linear(nn.Linear, LoraLayer):
    """在原 nn.Linear 旁增加低秩分支 B @ A。"""

    def __init__(
        self,
        in_features: int,
        out_features: int,
        r: int = 0,
        lora_alpha: int = 1,
        lora_dropout: float = 0.0,
        fan_in_fan_out: bool = False,
        merge_weights: bool = True,
        **kwargs,
    ):
        nn.Linear.__init__(self, in_features, out_features, **kwargs)
        LoraLayer.__init__(self, r=r, lora_alpha=lora_alpha,
                           lora_dropout=lora_dropout, merge_weights=merge_weights)
        self.fan_in_fan_out = fan_in_fan_out

        if r > 0:
            # A: (in_features -> r)，B: (r -> out_features)
            self.lora_A = nn.Linear(in_features, r, bias=False)
            self.lora_B = nn.Linear(r, out_features, bias=False)
            # 缩放系数，保持不同 r 下的更新幅度稳定
            self.scaling = self.lora_alpha / self.r
            # 冻结原权重，只训练 A/B
            self.weight.requires_grad = False

        self.reset_parameters()
        if fan_in_fan_out:
            self.weight.data = self.weight.data.T

    def reset_parameters(self):
        nn.Linear.reset_parameters(self)
        if hasattr(self, "lora_A"):
            nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B.weight)  # B=0，保证初始 BA=0

    def forward(self, x: torch.Tensor):
        if self.disable_adapters:
            if self.r > 0 and self.merged:
                # 关闭适配器前，先把已合并的 BA 减回去
                self.weight.data -= (
                    transpose(self.lora_B.weight @ self.lora_A.weight, self.fan_in_fan_out) * self.scaling
                )
                self.merged = False
            return F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)

        elif self.r > 0 and not self.merged:
            # 原路径 + 低秩旁路
            result = F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)
            result += self.lora_B(self.lora_A(self.lora_dropout(x))) * self.scaling
            return result

        else:
            return F.linear(x, transpose(self.weight, self.fan_in_fan_out), bias=self.bias)

In [ ]:
# 小例子：输入 8 维，输出 4 维，秩 r=2
layer = Linear(8, 4, r=2, lora_alpha=4)
x = torch.randn(3, 8)
y = layer(x)

print("输出形状:", tuple(y.shape))
print("scaling:", layer.scaling)
print("原权重可训练:", layer.weight.requires_grad)
print("可训练参数:", [n for n, p in layer.named_parameters() if p.requires_grad])
print("LoRA 参数量:", sum(p.numel() for n, p in layer.named_parameters() if "lora_" in n))

合并（merge）与推理：

- 训练时保持 `W0` 与 `BA` 分离；
- 推理前可执行 `W0 ← W0 + B @ A * scaling`，之后直接走原 `F.linear`；
- 因此 LoRA 在部署阶段没有额外计算延迟；
- 多个下游任务可以各存一份 `A/B`，共享同一份基座权重。

### 6.3.4 使用 peft 实现 LoRA

`peft` 把上述“查找目标层 → 替换为 LoRA 层 → 冻结原参数”的流程封装成：

```python
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)
model = get_peft_model(model, peft_config)
```

`target_modules` 可以是 `["q_proj", "v_proj"]`，也可以由 peft 自动推断。训练仍使用 `Trainer`，但显存占用显著降低。

⚠️ 以下 peft 代码需要安装 `peft` 并准备模型路径。

> 原文片段的 `MODEL_PATH` 未给出赋值语句，运行前请先把它设为本地基座模型路径，例如 `MODEL_PATH = "autodl-tmp/qwen-1.5b"`。

In [ ]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from transformers import Trainer

In [ ]:
# 加载基座模型
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_PATH, trust_remote_code=True
)

In [ ]:
peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=8,
            lora_alpha=32,
            lora_dropout=0.1,
        )

In [ ]:
model = get_peft_model(model, peft_config)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset= IterableWrapper(train_dataset),
    tokenizer=tokenizer
)
trainer.train()

注意：LoRA 只调整低秩子空间，适合任务适配，不擅长注入大量新知识；如果需要学习新知识，仍应考虑全量预训练 / 继续预训练。

## 6.4 偏好对齐

偏好对齐（Preference Alignment）承接在 SFT 之后，目标不是继续拟合文本，而是让模型在多个候选回答中更偏向人类喜欢的输出：更清晰、更安全、更有帮助、推理更稳定。

| 方法 | 是否需要奖励模型 | 核心思想 |
| --- | --- | --- |
| RLHF（PPO） | 需要 | 用人类偏好训练奖励模型，再用强化学习优化策略 |
| DPO | 不需要 | 直接用偏好对（chosen / rejected）构造损失优化模型 |
| KTO | 不需要 | 让模型对“好回答”和“不好回答”形成稳定区分 |

工程上，偏好对齐可以复用本章的能力：

- 模型加载、数据管线、`Trainer` / DeepSpeed 配置来自 6.1；
- SFT 模型通常作为偏好对齐的起点；
- LoRA / QLoRA / peft 同样可用于 DPO、KTO，降低显存开销。

因此，偏好对齐真正新增的内容主要是：**偏好数据组织、奖励建模方式、偏好损失函数**。初学者建议先从 DPO 入手，再理解完整的 RLHF 链路。

## 附：Notebook 与源码文件对应关系

| Notebook 部分 | 对应源码 |
| --- | --- |
| 6.1.2 初始化 LLM | 原文 6.1.2 片段 |
| 6.1.3 数据处理 | 原文 6.1.3 片段 |
| 6.1.4 Trainer | 原文 6.1.4 片段 |
| 6.1.5 DeepSpeed | `code/pretrain.py`、`code/ds_config_zero2.json`、`code/pretrain.sh` |
| 6.2.2 SFT 数据 | `code/finetune.py` 中的 `preprocess` / `SupervisedDataset` |
| 6.2.3 SFT 训练 | `code/finetune.py` 的 `main()`、`code/finetune.sh` |
| 6.3.4 LoRA | 原文 6.3.4 片段（补全后可运行） |
| 6.3.5 peft | 原文 6.3.5 片段 |

补充说明：

- `RUN_PRETRAIN` / `RUN_TRAINING` 是 Notebook 为防误执行新增的开关，原始脚本对应 `if __name__ == "__main__":`。
- `6.1.2`～`6.1.4` 的片段按原文顺序保留，单独运行时需要先准备模型、Tokenizer 与数据。
- LoRA 一节把原文拆开的 `LoraLayer`、`Linear`、`forward` 合并，并补上了 `transpose` 与 `reset_parameters`，因此可以直接运行。
- 原文 `code/finetune.py` 中 `IGNORE_TOKEN_ID` 使用 `pad_token_id`，注释里也提到主流实现常用 `-100`；两者只要与损失函数的 `ignore_index` 一致即可。